# Lab 21-07 · Power Amplifiers

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pa


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Conduction angle

*Predict* the efficiency at 360°, 240°, 180° and 120°.

In [2]:
alphas = np.linspace(10, 360, 351)
eta = [pa.conduction(math.radians(a))[2] * 100 for a in alphas]
prel = [20 * math.log10(pa.conduction(math.radians(a))[1] / 0.5) for a in alphas]
for a in (360, 240, 180, 120):
    idc, i1, e = pa.conduction(math.radians(a))
    print(f"{a:3d} deg: efficiency {e*100:5.1f} %, power vs class A {20*math.log10(i1/0.5):+.2f} dB")
fig, ax = plt.subplots(); ax.plot(alphas, eta); ax.set_xlabel("conduction angle (degrees)"); ax.set_ylabel("efficiency (%)")
ax2 = ax.twinx(); ax2.plot(alphas, prel, "--", color="C1"); ax2.set_ylabel("power vs class A (dB)"); plt.show()

360 deg: efficiency  50.0 %, power vs class A +0.00 dB
240 deg: efficiency  66.1 %, power vs class A +0.61 dB
180 deg: efficiency  78.5 %, power vs class A +0.00 dB
120 deg: efficiency  89.7 %, power vs class A -2.14 dB


## Part 2 · Class E in ngspice

*Predict* the output power, efficiency and peak switch voltage of Example 21-07.3.

In [3]:
s = ngspice("classe.cir")
print({k: round(s[k], 4) for k in ("pout", "pin", "eff", "vpk")})
ce = pa.class_e()
print(f"ideal: 1.000 W, 100 %, {ce['vpk']*3.3:.2f} V")
# ideal waveforms from the closed-form solution (V_DD = 1, I_DC = 1)
th = np.linspace(0, 4 * np.pi, 2000)
phi, im = math.radians(ce["phi_deg"]), ce["Im_Idc"]
ph = th % (2 * np.pi)
v = np.where(ph >= np.pi, ((ph - np.pi) + im * (np.cos(ph + phi) - np.cos(np.pi + phi))) * math.pi, 0.0)
i = np.where(ph < np.pi, 1 - im * np.sin(ph + phi), 0.0)
plt.figure(); plt.plot(th / (2 * np.pi), v, label="switch voltage / V_DD"); plt.plot(th / (2 * np.pi), i, label="switch current / I_DC")
plt.xlabel("cycles"); plt.legend(); plt.show()

{'pout': 1.0132, 'pin': 1.0137, 'eff': 0.9995, 'vpk': 11.9373}
ideal: 1.000 W, 100 %, 11.75 V


## Part 3 · Efficiency in back-off, and averages over an OFDM envelope

*Predict* each architecture's efficiency at 6 and 10 dB back-off.

In [4]:
x = np.linspace(0.005, 1, 400)
bo = -20 * np.log10(x)
curves = {"class A": [pa.eta_backoff("A", t) for t in x], "class B": [pa.eta_backoff("B", t) for t in x],
          "Doherty": [pa.eta_doherty(t) for t in x], "ET (20 % floor, 85 % modulator)": [0.85 * math.pi / 4 * t / max(t, 0.2) for t in x]}
papr = 10 * math.log10(math.log(1e4))
fig, ax = plt.subplots()
for name, c in curves.items():
    ax.plot(bo, np.array(c) * 100, label=name)
u = np.linspace(0, 25, 400)
z = 10 ** ((papr - u) / 10)                       # envelope power relative to average, at back-off u from the peak
ax2 = ax.twinx(); ax2.fill_between(u, 0, (np.log(10) / 10) * z * np.exp(-z), alpha=0.15, color="gray")
ax2.set_ylabel("OFDM envelope density (per dB)")
ax.set_xlim(0, 25); ax.set_xlabel("output back-off from peak (dB)"); ax.set_ylabel("efficiency (%)"); ax.legend(fontsize=8); plt.show()
for name, fn in (("class A", lambda t: pa.eta_backoff("A", t)), ("class B", lambda t: pa.eta_backoff("B", t)),
                 ("Doherty", pa.eta_doherty), ("ET PA", lambda t: math.pi / 4 * t / max(t, 0.2))):
    print(f"{name:8s} average efficiency for a 9.64 dB-PAPR OFDM signal: {pa.average_efficiency(fn, papr)*100:5.2f} %")
print(f"ET overall with an 85 % modulator: {0.85*pa.average_efficiency(lambda t: math.pi/4*t/max(t, 0.2), papr)*100:5.2f} %")

class A  average efficiency for a 9.64 dB-PAPR OFDM signal:  5.43 %
class B  average efficiency for a 9.64 dB-PAPR OFDM signal: 29.20 %
Doherty  average efficiency for a 9.64 dB-PAPR OFDM signal: 54.90 %
ET PA    average efficiency for a 9.64 dB-PAPR OFDM signal: 77.03 %
ET overall with an 85 % modulator: 65.48 %


## Part 4 · The Doherty in ngspice

*Predict* the main device's swing and the efficiency at x = 0.5, 2/3 and 1.

In [5]:
s = ngspice("doherty.cir")
print({k: round(s[k], 4) for k in ("vma", "vmc", "poa", "pob", "poc", "effa", "effb", "effc")})
print("theory:", [round(pa.eta_doherty(t), 4) for t in (0.5, 2 / 3, 1.0)])

{'vma': 27.9964, 'vmc': 27.9975, 'poa': 7.8376, 'pob': 13.9344, 'poc': 31.3546, 'effa': 0.7852, 'effb': 0.698, 'effc': 0.7853}
theory: [0.7854, 0.6981, 0.7854]


## Part 5 · ACLR of a cubic PA

*Predict* the ACLR for $a_3$ = −0.05 at an input power of 0.01 (−20 dB).

In [6]:
x = pa.ofdm()
P = 0.01
y = pa.cubic(math.sqrt(P) * x, 1.0, -0.05)
print(f"simulated {pa.aclr_db(y):.2f} dBc, formula {10*math.log10(0.05**2*P**2/3):.2f} dBc")
seg = y.reshape(-1, 1024)
S = np.fft.fftshift(np.mean(np.abs(np.fft.fft(seg, axis=1)) ** 2, axis=0))
f = np.arange(-512, 512)
plt.figure(); plt.plot(f, 10 * np.log10(S / S.max())); plt.xlim(-300, 300); plt.ylim(-110, 5)
plt.xlabel("frequency (bins; channel = 128 bins)"); plt.ylabel("PSD (dB)"); plt.show()

simulated -70.45 dBc, formula -70.79 dBc


## Part 6 · Digital predistortion

*Predict* whether DPD helps more with 12 dB or 10 dB of margin between average power and saturation.

In [7]:
plt.figure()
for margin in (12, 10):
    asat = 10 ** (margin / 20)
    amp = lambda s_, a=asat: pa.rapp(s_, 1.0, a, 2.0)
    y0 = amp(x)
    u = pa.dpd_loop(x, amp, iterations=4, orders=(1, 3, 5))
    y1 = amp(u)
    a = np.minimum(np.abs(x), 0.98 * asat)
    y2 = amp(a / (1 - (a / asat) ** 4) ** 0.25 * np.exp(1j * np.angle(x)))
    print(f"margin {margin} dB: ACLR none {pa.aclr_db(y0):.1f}, DPD {pa.aclr_db(y1):.1f}, ideal inverse {pa.aclr_db(y2):.1f} dBc; "
          f"EVM none {pa.evm_pct(y0, x):.2f} %, DPD {pa.evm_pct(y1, x):.2f} %")
    if margin == 12:
        for yy, lab_ in ((y0, "no DPD"), (y1, "5th-order DPD")):
            S = np.fft.fftshift(np.mean(np.abs(np.fft.fft(yy.reshape(-1, 1024), axis=1)) ** 2, axis=0))
            plt.plot(np.arange(-512, 512), 10 * np.log10(S / S.max()), label=f"12 dB margin, {lab_}")
plt.xlim(-300, 300); plt.ylim(-110, 5); plt.xlabel("frequency (bins)"); plt.ylabel("PSD (dB)"); plt.legend(); plt.show()

margin 12 dB: ACLR none -48.6, DPD -66.4, ideal inverse -318.7 dBc; EVM none 0.88 %, DPD 0.10 %
margin 10 dB: ACLR none -41.8, DPD -48.3, ideal inverse -55.2 dBc; EVM none 1.94 %, DPD 0.79 %


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.